# Phishing URL Triage Dashboard

This notebook builds an interactive dashboard from `reports/report.csv`.

Usage:
- Run `triage.py` to create `reports/report.csv`.
- Run the cells in order.

## 1. Install and Import Libraries for Dashboarding

In [ ]:
# If needed, install dependencies
# !pip install -q pandas plotly dash jupyter-dash

from pathlib import Path

import pandas as pd
import plotly.express as px
from dash import Input, Output, State, dcc, html
from dash import dash_table
from jupyter_dash import JupyterDash

## 2. Load and Validate Data for the Dashboard

In [ ]:
report_path = Path("reports") / "report.csv"

if not report_path.exists():
    raise FileNotFoundError(
        "report.csv not found. Run triage.py first to generate reports/report.csv."
    )

df = pd.read_csv(report_path)

required_cols = {"url", "score", "risk", "signals"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

df["url"] = df["url"].fillna("")
df["signals"] = df["signals"].fillna("")
df["risk"] = df["risk"].fillna("unknown").str.lower()
df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0).astype(int)

df.head()

## 3. Prepare Aggregations and KPIs

In [ ]:
def compute_aggregations(dataframe: pd.DataFrame):
    risk_counts = dataframe["risk"].value_counts().reset_index()
    risk_counts.columns = ["risk", "count"]

    signal_names = []
    for value in dataframe["signals"]:
        for chunk in str(value).split(";"):
            chunk = chunk.strip()
            if not chunk:
                continue
            name = chunk.split(":")[0].strip()
            if name:
                signal_names.append(name)

    if signal_names:
        signal_counts = (
            pd.Series(signal_names)
            .value_counts()
            .reset_index()
            .rename(columns={"index": "signal", 0: "count"})
        )
    else:
        signal_counts = pd.DataFrame({"signal": [], "count": []})

    top10 = dataframe.sort_values("score", ascending=False).head(10)

    kpis = {
        "total": int(len(dataframe)),
        "high": int((dataframe["risk"] == "high").sum()),
        "medium": int((dataframe["risk"] == "medium").sum()),
        "low": int((dataframe["risk"] == "low").sum()),
        "avg_score": float(round(dataframe["score"].mean(), 2)) if len(dataframe) else 0.0,
    }

    return risk_counts, signal_counts, top10, kpis

risk_counts, signal_counts, top10, kpis = compute_aggregations(df)

kpis

## 4. Build an Interactive Dashboard Layout

In [ ]:
risk_bar_fig = px.bar(risk_counts, x="risk", y="count", title="Risk Distribution (Bar)")
risk_pie_fig = px.pie(risk_counts, names="risk", values="count", title="Risk Distribution (Pie)")

if not signal_counts.empty:
    top_signals_fig = px.bar(
        signal_counts.head(10), x="signal", y="count", title="Top Signals"
    )
else:
    top_signals_fig = px.bar(
        pd.DataFrame({"signal": ["none"], "count": [0]}),
        x="signal",
        y="count",
        title="Top Signals",
    )

app = JupyterDash(__name__)

risk_options = ["all"] + sorted(df["risk"].unique().tolist())

kpi_style = {
    "display": "inline-block",
    "padding": "12px 16px",
    "margin": "6px",
    "border": "1px solid #ccc",
    "borderRadius": "6px",
    "minWidth": "120px",
    "textAlign": "center",
}

control_style = {
    "padding": "10px 12px",
    "border": "1px solid #ddd",
    "borderRadius": "8px",
    "marginBottom": "12px",
}

app.layout = html.Div(
    [
        html.H2("Phishing URL Triage Dashboard"),
        html.Div(
            [
                html.Div([html.Div("Total"), html.H4(kpis["total"], id="kpi-total")], style=kpi_style),
                html.Div([html.Div("High"), html.H4(kpis["high"], id="kpi-high")], style=kpi_style),
                html.Div([html.Div("Medium"), html.H4(kpis["medium"], id="kpi-medium")], style=kpi_style),
                html.Div([html.Div("Low"), html.H4(kpis["low"], id="kpi-low")], style=kpi_style),
                html.Div(
                    [html.Div("Avg Score"), html.H4(kpis["avg_score"], id="kpi-avg")],
                    style=kpi_style,
                ),
            ]
        ),
        html.Div(
            [
                html.Label("Filter by risk"),
                dcc.Dropdown(
                    id="risk-filter",
                    options=[{"label": value, "value": value} for value in risk_options],
                    value="all",
                    clearable=False,
                ),
                html.Label("Minimum score"),
                dcc.Slider(id="min-score", min=0, max=80, step=5, value=0),
                html.Label("Search URL or signal"),
                dcc.Input(
                    id="search-query",
                    type="text",
                    placeholder="paypal, verify, 192.168",
                    style={"width": "100%", "marginBottom": "8px"},
                ),
                html.Button("Download filtered CSV", id="download-csv", n_clicks=0),
                dcc.Download(id="download-data"),
            ],
            style=control_style,
        ),
        dcc.Graph(id="risk-bar", figure=risk_bar_fig),
        dcc.Graph(id="risk-pie", figure=risk_pie_fig),
        dcc.Graph(id="signals-bar", figure=top_signals_fig),
        html.H3("Top 10 Highest Scores"),
        dash_table.DataTable(
            id="top-table",
            columns=[
                {"name": "url", "id": "url"},
                {"name": "score", "id": "score"},
                {"name": "risk", "id": "risk"},
                {"name": "signals", "id": "signals"},
            ],
            data=top10.to_dict("records"),
            page_size=10,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "fontFamily": "Arial", "fontSize": 12},
        ),
    ],
    style={"maxWidth": "1100px", "margin": "0 auto"},
)

app

## 5. Wire Up Filters and Callbacks

In [ ]:
def filter_dataframe(selected_risk, min_score, query):
    filtered = df.copy()

    if selected_risk and selected_risk != "all":
        filtered = filtered[filtered["risk"] == selected_risk]

    if min_score is not None:
        filtered = filtered[filtered["score"] >= int(min_score)]

    if query:
        lowered = query.strip().lower()
        if lowered:
            mask = (
                filtered["url"].str.lower().str.contains(lowered, na=False)
                | filtered["signals"].str.lower().str.contains(lowered, na=False)
            )
            filtered = filtered[mask]

    return filtered


@app.callback(
    Output("risk-bar", "figure"),
    Output("risk-pie", "figure"),
    Output("signals-bar", "figure"),
    Output("top-table", "data"),
    Output("kpi-total", "children"),
    Output("kpi-high", "children"),
    Output("kpi-medium", "children"),
    Output("kpi-low", "children"),
    Output("kpi-avg", "children"),
    Input("risk-filter", "value"),
    Input("min-score", "value"),
    Input("search-query", "value"),
)
def update_dashboard(selected_risk, min_score, query):
    filtered = filter_dataframe(selected_risk, min_score, query)

    local_risk_counts, local_signal_counts, local_top10, local_kpis = compute_aggregations(
        filtered
    )

    if local_risk_counts.empty:
        local_risk_counts = pd.DataFrame({"risk": ["none"], "count": [0]})
    if local_signal_counts.empty:
        local_signal_counts = pd.DataFrame({"signal": ["none"], "count": [0]})

    bar_fig = px.bar(
        local_risk_counts, x="risk", y="count", title="Risk Distribution (Bar)"
    )
    pie_fig = px.pie(
        local_risk_counts, names="risk", values="count", title="Risk Distribution (Pie)"
    )

    signal_fig = px.bar(
        local_signal_counts.head(10),
        x="signal",
        y="count",
        title="Top Signals",
    )

    return (
        bar_fig,
        pie_fig,
        signal_fig,
        local_top10.to_dict("records"),
        local_kpis["total"],
        local_kpis["high"],
        local_kpis["medium"],
        local_kpis["low"],
        local_kpis["avg_score"],
    )


@app.callback(
    Output("download-data", "data"),
    Input("download-csv", "n_clicks"),
    State("risk-filter", "value"),
    State("min-score", "value"),
    State("search-query", "value"),
    prevent_initial_call=True,
)
def download_filtered(n_clicks, selected_risk, min_score, query):
    filtered = filter_dataframe(selected_risk, min_score, query)
    return dcc.send_data_frame(filtered.to_csv, "filtered_report.csv", index=False)


update_dashboard

## 6. Render and Export Dashboard Outputs

In [ ]:
export_dir = Path("dashboard_exports")
export_dir.mkdir(exist_ok=True)

risk_bar_fig.write_html(export_dir / "risk_bar.html")
risk_pie_fig.write_html(export_dir / "risk_pie.html")
top_signals_fig.write_html(export_dir / "top_signals.html")

print(f"Exported HTML charts to {export_dir}")

app.run_server(mode="inline", height=750)